# 05 · FT-Transformer — Quantile Preprocessing (Lending Club)

Single experiment: apply `QuantileTransformer(output_distribution="normal")` to all numerical features before standard scaling and feed the transformed values to a vanilla FT-Transformer. The hypothesis is that mapping heavy-tailed numerical distributions to approximately Gaussian inputs improves transformer attention quality on tabular data.

Architecture: standard FTTransformer with `d_token=64`, `n_blocks=5`, `n_heads=4`. The model itself is unchanged from `01_ftt_vanilla_hyperopt`; only the numerical preprocessing differs.

## 1. Setup

In [ ]:
%pip install rtdl_revisiting_models -q
%pip install delu -q
%pip install optuna -q

In [ ]:
import math
import random
import warnings
import json
import os
import itertools
import typing
from collections import OrderedDict
from pathlib import Path
from typing import Any, Dict, Iterable, List, Literal, Optional, Tuple, cast

import numpy as np
import pandas as pd
import scipy.special

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim
from torch import Tensor
from torch.nn.parameter import Parameter
from torch.utils.data import Dataset, DataLoader, TensorDataset

import sklearn.datasets
import sklearn.metrics
import sklearn.model_selection
import sklearn.preprocessing
import sklearn.tree as sklearn_tree
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder, QuantileTransformer
from sklearn.metrics import roc_auc_score
from sklearn.isotonic import IsotonicRegression

import optuna
import delu
from tqdm.std import tqdm
from tqdm import tqdm as _tqdm  # alias for cells that use it directly

warnings.filterwarnings("ignore")
pd.options.display.max_rows = 200
pd.options.display.max_columns = 200

RANDOM_SEED = 42


def seed_everything(seed: int = RANDOM_SEED) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


seed_everything()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

In [ ]:
from rtdl_revisiting_models import FTTransformer

In [ ]:
# --- Optional: Google Colab Drive mount ---
# Uncomment the three lines below if you're running on Colab and want to read
# data from / write artifacts to your Drive. Skip on local, server, or Kaggle
# runs. The next cell auto-routes through DRIVE_ROOT when defined.
#
# from google.colab import drive
# drive.mount("/content/drive")
# DRIVE_ROOT = "/content/drive/MyDrive/ft-transformer-credit-risk-study"

# When running locally, repo root is one directory up (notebook is in
# notebooks/<dataset>/). On Colab with the cell above uncommented, DRIVE_ROOT
# takes precedence.
_BASE = globals().get("DRIVE_ROOT", "..")
DATA_PATH      = f"{_BASE}/data/processed_data_densest.parquet.gzip"
ARTIFACTS_DIR  = Path(f"{_BASE}/artifacts/lending_club")
RESULTS_DIR    = ARTIFACTS_DIR / "results"
MODELS_DIR     = ARTIFACTS_DIR / "models"
FIGURES_DIR    = ARTIFACTS_DIR / "figures"
for _d in (ARTIFACTS_DIR, RESULTS_DIR, MODELS_DIR, FIGURES_DIR):
    _d.mkdir(parents=True, exist_ok=True)

print(f"DATA_PATH      = {DATA_PATH}")
print(f"ARTIFACTS_DIR  = {ARTIFACTS_DIR}")

In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# DEV_MODE — smoke-test switch.
#
# When False (the default), every constant resolves to the exact value used in
# the original experiment. The DEV_MODE branches below are dead code in that
# path, so the run is behaviourally identical to the source notebook.
#
# When True, the notebook runs a tiny fast version: a subsample of the data,
# a couple of epochs, a single Optuna trial. Use this on Colab to validate the
# full pipeline end-to-end before launching a real run.
# ──────────────────────────────────────────────────────────────────────────────
DEV_MODE = False
if DEV_MODE:
    print("DEV_MODE is ON — smoke-test run with reduced data/epochs/trials.")

## 2. Data Loading

In [ ]:
# Load preprocessed Lending Club dataset.
df4 = pd.read_parquet(DATA_PATH)

train_val_df, test_df = train_test_split(
    df4, test_size=0.2,
    stratify=df4["target_binary"], random_state=RANDOM_SEED,
)
train_df, val_df = train_test_split(
    train_val_df, test_size=0.2,
    stratify=train_val_df["target_binary"], random_state=RANDOM_SEED,
)
print("Train_val / test sizes:", len(train_val_df), len(test_df))
print("Train / val sizes:", len(train_df), len(val_df))

cat_columns = [
    "term", "emp_length", "home_ownership", "verification_status",
    "purpose", "zip_code", "addr_state", "application_type",
    "initial_list_status", "disbursement_method",
]
num_cols = [
    c for c in list(df4.columns)
    if c not in cat_columns + ["id", "emp_title", "target_binary"]
]
print(f"# numerical features: {len(num_cols)}  |  # categorical: {len(cat_columns)}")

## 3. Preprocessing

Numeric pipeline: median imputation → `QuantileTransformer` (normal output) → `StandardScaler`. Categorical pipeline uses label encoding with an `UNKNOWN` bucket.

In [ ]:
# Numeric pipeline: median impute -> QuantileTransformer (normal output) -> StandardScaler.
num_impute = {}
qt = QuantileTransformer(output_distribution="normal", random_state=RANDOM_SEED)
scaler = StandardScaler()

num_df = train_df[num_cols].copy()
for c in num_df.columns:
    if num_df[c].dtype == "object":
        num_df[c] = pd.to_numeric(num_df[c], errors="coerce")
    med = num_df[c].median()
    num_df[c] = num_df[c].fillna(med)
    num_impute[c] = med

num_df[num_df.columns] = qt.fit_transform(num_df[num_df.columns])
num_df[num_df.columns] = scaler.fit_transform(num_df[num_df.columns])

cat_encoders: Dict[str, LabelEncoder] = {}
cat_cardinalities = []
cat_df = train_df[cat_columns].copy()
for c in cat_columns:
    le = LabelEncoder()
    series = cat_df[c].fillna("MISSING").astype(str)
    le.fit(series)
    new_classes = list(le.classes_)
    if "MISSING" not in new_classes:
        new_classes.append("MISSING")
    new_classes.append("UNKNOWN")
    le.classes_ = np.array(new_classes)
    cat_df[c + "_le"] = le.transform(series)
    cat_encoders[c] = le
    cat_cardinalities.append(len(le.classes_))

train_df_proc = pd.concat(
    [
        num_df.reset_index(drop=True),
        cat_df[[c + "_le" for c in cat_columns]].reset_index(drop=True),
        train_df["target_binary"].reset_index(drop=True),
    ],
    axis=1,
)


def _apply_transforms_quantile(part_df):
    num_part = part_df[num_cols].copy()
    for c in num_part.columns:
        if num_part[c].dtype == "object":
            num_part[c] = pd.to_numeric(num_part[c], errors="coerce")
    for c in num_part.columns:
        num_part[c] = num_part[c].fillna(num_impute[c])
    num_part = pd.DataFrame(
        qt.transform(num_part), columns=num_part.columns, index=num_part.index,
    )
    num_part = pd.DataFrame(
        scaler.transform(num_part), columns=num_part.columns, index=num_part.index,
    )

    cat_part = part_df[cat_columns].copy()
    for c in cat_columns:
        le = cat_encoders[c]
        mapping = {cls: i for i, cls in enumerate(le.classes_)}
        unknown_id = mapping["UNKNOWN"]
        series = cat_part[c].fillna("MISSING").astype(str)
        cat_part[c + "_le"] = series.map(mapping).fillna(unknown_id).astype(int)

    return pd.concat(
        [
            num_part.reset_index(drop=True),
            cat_part[[c + "_le" for c in cat_columns]].reset_index(drop=True),
            part_df["target_binary"].reset_index(drop=True),
        ],
        axis=1,
    )


val_df_proc  = _apply_transforms_quantile(val_df)
test_df_proc = _apply_transforms_quantile(test_df)
print("Encoded train / val / test with QuantileTransformer applied to numerics.")

In [ ]:
# Materialize everything as torch tensors on the target device.
cat_le_cols = [c + "_le" for c in cat_columns]
data_numpy = {"train": {}, "val": {}, "test": {}}
for part_name, part_df in [
    ("train", train_df_proc),
    ("val",   val_df_proc),
    ("test",  test_df_proc),
]:
    data_numpy[part_name]["X_cont"] = part_df[num_cols]
    data_numpy[part_name]["x_cat"]  = part_df[cat_le_cols]
    data_numpy[part_name]["y"]      = part_df["target_binary"]

data = {}
for part in data_numpy:
    data[part] = {
        "X_cont": torch.tensor(data_numpy[part]["X_cont"].to_numpy(), dtype=torch.float32, device=device),
        "x_cat":  torch.tensor(data_numpy[part]["x_cat"].to_numpy(),  dtype=torch.long,    device=device),
        "y":      torch.tensor(data_numpy[part]["y"].to_numpy(),      dtype=torch.float32, device=device),
    }

d_numerical = len(num_cols)
n_train = data["train"]["y"].shape[0]
print(f"d_numerical = {d_numerical}")
print(f"cat_cardinalities = {cat_cardinalities}")
print(f"# train rows = {n_train}")

In [ ]:
# When DEV_MODE is on, subsample data to a few thousand rows so a full training
# pass completes in a couple of minutes. No-op when DEV_MODE is False.
if DEV_MODE:
    _n_dev = 5000
    for _part in data:
        _idx = torch.randperm(data[_part]["y"].shape[0], device=device)[:_n_dev]
        for _k in data[_part]:
            data[_part][_k] = data[_part][_k][_idx]
    print({_part: {k: v.shape for k, v in data[_part].items()} for _part in data})

## 4. Optuna Optimization

In [ ]:
import json
trial_results = [] # stores AUC + hyperparams
global_best_auc=0.73
global_model_path = str(MODELS_DIR / "ftt_quantile_lc.pth")
d_out=1
BATCH_SIZE = 4192+4192
from rtdl_revisiting_models import FTTransformer
import math
def optim(trial):
    global global_best_auc
    # -----------------------
    # Search Space - shallow nets
    # -----------------------
    d_Block = trial.suggest_categorical("d_token",[64]) #
    n_Blocks = trial.suggest_int("n_blocks", 5,5)
    Attention_n_heads = trial.suggest_categorical("attention_n_heads", [4]) #
    FFN_d_hidden = trial.suggest_categorical("ffn_d_hidden", [8/3])
    Lr = trial.suggest_float("lr", 2e-4,2e-4, log=True)# 2e-3

    # Print trial parameters at the start of each trial
    print("\n" + "="*70)
    print(f" Starting Trial {trial.number}")
    print(" Hyperparameters:")
    print(f" d_token = {d_Block}")
    print(f" n_blocks = {n_Blocks}")
    print(f" attention_n_heads = {Attention_n_heads}")
    print(f" ffn_d_hidden = {FFN_d_hidden}")
    print(f" lr = {Lr}")
    print("="*70 + "\n")

    model = FTTransformer(
        n_cont_features=d_numerical,
        cat_cardinalities=cat_cardinalities,
        n_blocks=n_Blocks,
        d_block=d_Block,
        attention_n_heads=Attention_n_heads,
        ffn_d_hidden_multiplier=FFN_d_hidden,#
        attention_dropout=0.05,#Attention_dropout
        ffn_dropout=0.05,#FFN_dropout
        residual_dropout=0.05,#Residual_droupout
        d_out=1 # output a single logit (binary)
    ).to(device)
    n_epochs = 2 if DEV_MODE else 100
    patience = 10

    # model = torch.nn.DataParallel(model, device_ids = [0,1]).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=Lr, weight_decay=1e-5)
    scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer,
    max_lr=Lr * 5, # Recommended: 3–10× base LR
    steps_per_epoch=math.ceil(train_df_proc.shape[0] / BATCH_SIZE), # NUMBER OF BATCHES PER EPOCH
    epochs=n_epochs,
    pct_start=0.3,
    anneal_strategy="cos"
)

    loss_fn = F.binary_cross_entropy_with_logits

    def apply_model(batch: Dict[str, Tensor]) -> Tensor:
            return model(batch["X_cont"], batch.get("x_cat")).squeeze(-1)

    @torch.no_grad()
    def evaluate(part: str) -> float:
        model.eval()
        eval_batch_size = BATCH_SIZE
        y_pred = (
            torch.cat(
                [
                    apply_model(batch)
                    for batch in delu.iter_batches(data[part], eval_batch_size)
                ]
            )
            .cpu()
            .numpy()
        )
        y_true = data[part]["y"].cpu().numpy()
        y_pred = scipy.special.expit(y_pred)
        auc = roc_auc_score(y_true, y_pred)
        return auc # The higher -- the better.


    print(f'Test score before training: AUC Val = {evaluate("val"):.4f}')


    batch_size = BATCH_SIZE
    epoch_size = math.ceil(train_df.shape[0] / batch_size)
    timer = delu.tools.Timer()
    early_stopping = delu.tools.EarlyStopping(patience, mode="max")
    best = {
        "val": -math.inf,
        "test": -math.inf,
        "epoch": -1,
    }

    print(f"Device: {device.type.upper()}")
    print("-" * 88 + "\n")
    timer.run()
    for epoch in range(n_epochs):
        for batch in tqdm(
            delu.iter_batches(data["train"], batch_size, shuffle=True),
            desc=f"Epoch {epoch}",
            total=epoch_size,
        ):
            model.train()
            optimizer.zero_grad()

            y_batch = batch["y"].float()
            weights = torch.where(
                y_batch == 1,
                torch.tensor(3.5, device=y_batch.device),
                torch.tensor(1.0, device=y_batch.device)
            )

            loss = loss_fn(apply_model(batch), batch["y"].float(), weight=weights)
            loss.backward()
            optimizer.step()
            scheduler.step()

        val_auc = evaluate("val")
        test_auc = evaluate("test")
        # train_auc = evaluate('train')
        # print(f"(val) {val_score:.4f} (test) {test_score:.4f} [time] {timer}")
        print(f" AUC Val {val_auc:.4f} Test {test_auc:.4f} [time] {timer}")
        # print(f" AUC Train {train_auc:.4f} Val {val_auc:.4f} (test) {test_auc:.4f} [time] {timer}")

        early_stopping.update(val_auc)
        if early_stopping.should_stop():
            break

        if val_auc > best["val"]:
            print(" New best epoch! ")
            best = {"val": val_auc, "test": test_auc, "epoch": epoch}
            if val_auc>global_best_auc:
              torch.save(model.state_dict(), global_model_path)
              global_best_auc=val_auc

        # print()

    trial_results.append({
        "trial_type": 'QT numerical columns',
        "auc_val": val_auc,
        "auc_test": test_auc,
        "d_token": d_Block,
        "n_blocks": n_Blocks,
        "n_heads": Attention_n_heads,
        "ffn_d_hidden": FFN_d_hidden,
        "attention_dropout": 0.05, #Attention_dropout,
        "ffn_dropout": 0.05, #FFN_dropout,
        'Residual_droupout':0.05, #Residual_droupout,
        'Lr':Lr,
        'best':best,
        'time':timer.__str__()
    })
    with open(str(RESULTS_DIR / "ftt_quantile_lc_trials.json"), "w") as f:
        json.dump(trial_results, f, indent=4)

    # print(trial_results[trial.number])
    print("\n\nResult:")
    print(best)
    return best['val']
study = optuna.create_study(direction="maximize")
study.optimize(optim, n_trials=(1 if DEV_MODE else 1))